In [ ]:

from pathlib import Path
import pandas as pd
import geopandas as gpd

BASE_DIR = Path.cwd()
RAW = BASE_DIR / "data" / "raw"

# 1. Taxi data
print("\n========== NYC TAXI ==========")

taxi = pd.read_parquet(
    RAW / "tlc" / "yellow_tripdata_2024-01.parquet"
)

print("Shape:", taxi.shape)
print("Columns:", taxi.columns.tolist())
print("\nMissing values:")
print(taxi.isnull().sum())


# 2. Census data
print("\n========== CENSUS ACS ==========")

census = pd.read_csv(
    RAW / "census" / "nyc_acs_2024.csv"
)

print("Shape:", census.shape)
print("Columns:", census.columns.tolist())
print("\nMissing values:")
print(census.isnull().sum())


# 3. Taxi zone boundaries
print("\n========== TAXI ZONES ==========")

zones = gpd.read_file(
    RAW / "boundaries" / "taxi_zones.geojson"
)

print("Shape:", zones.shape)
print("CRS:", zones.crs)
print("Columns:", zones.columns.tolist())
print("Geometry types:", zones.geom_type.value_counts())


# 4. Census boundaries
print("\n========== CENSUS BOUNDARIES ==========")

tracts = gpd.read_file(
    RAW / "boundaries" / "nyc_census_tracts.geojson"
)

print("Shape:", tracts.shape)
print("CRS:", tracts.crs)
print("Geometry types:", tracts.geom_type.value_counts())


# 5. OSM data
print("\n========== OPENSTREETMAP ==========")

roads = gpd.read_file(
    RAW / "osm" / "nyc_roads.gpkg",
    layer="roads"
)

infrastructure = gpd.read_file(
    RAW / "osm" / "nyc_infrastructure.gpkg",
    layer="infrastructure"
)

print("Road features:", len(roads))
print("Road CRS:", roads.crs)

print("Infrastructure features:", len(infrastructure))
print("Infrastructure CRS:", infrastructure.crs)

print("\nDATA INSPECTION COMPLETE")


NameError: name '__file__' is not defined

In [2]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd


# --------------------------------------------------
# PATHS
# --------------------------------------------------
RAW = "data/raw"
PROCESSED = "data/processed"
os.makedirs(PROCESSED, exist_ok=True)

TAXI_ZONES = f"{RAW}/boundaries/taxi_zones.geojson"
CENSUS_BOUNDARIES = f"{RAW}/boundaries/nyc_census_tracts.geojson"
CENSUS_DATA = f"{RAW}/census/nyc_acs_2024.csv"   # change year if your file differs
OSM_ROADS = f"{RAW}/osm/nyc_roads.gpkg"
OSM_INFRA = f"{RAW}/osm/nyc_infrastructure.gpkg"

OUTPUT_CROSSWALK = f"{PROCESSED}/tract_zone_crosswalk.csv"
OUTPUT_FINAL = f"{PROCESSED}/taxi_zone_features.csv"

# UTM zone 18N: metres, accurate for NYC area/length calculations
PROJECTED_CRS = "EPSG:32618"


# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------
print("\nLoading datasets...")

zones = gpd.read_file(TAXI_ZONES)
tracts = gpd.read_file(CENSUS_BOUNDARIES)
census = pd.read_csv(CENSUS_DATA)
roads = gpd.read_file(OSM_ROADS)
infrastructure = gpd.read_file(OSM_INFRA)

# Make column names consistent regardless of which download script made the file
zones.columns = [c if c == "geometry" else c.lower() for c in zones.columns]

print("Taxi zone polygons:", len(zones))
print("Census tracts:", len(tracts))
print("Census records:", len(census))
print("OSM roads:", len(roads))
print("OSM infrastructure:", len(infrastructure))


# --------------------------------------------------
# 2. PREPARE CENSUS DATA
# --------------------------------------------------
print("\nPreparing Census data...")

census["GEOID"] = (
    census["state"].astype(str).str.zfill(2)
    + census["county"].astype(str).str.zfill(3)
    + census["tract"].astype(str).str.zfill(6)
)
tracts["GEOID"] = (
    tracts["STATEFP"].astype(str).str.zfill(2)
    + tracts["COUNTYFP"].astype(str).str.zfill(3)
    + tracts["TRACTCE"].astype(str).str.zfill(6)
)

census_columns = [
    "B01003_001E",  # Total population
    "B19013_001E",  # Median household income
    "B17001_001E",  # Poverty universe
    "B17001_002E",  # Population below poverty
    "B08201_001E",  # Total households
    "B08201_002E",  # Zero-vehicle households
    "B03002_001E",  # Total population for race/ethnicity
    "B03002_003E",  # Non-Hispanic White population
]

for col in census_columns:
    census[col] = pd.to_numeric(census[col], errors="coerce")
    # Census encodes "no data" as large negatives (e.g. -666666666).
    # Left in place they would corrupt every sum and average.
    census.loc[census[col] < 0, col] = np.nan

tracts = tracts.merge(census[["GEOID"] + census_columns], on="GEOID", how="left")
print("Tracts with a census match:", tracts["B01003_001E"].notna().sum(), "/", len(tracts))


# --------------------------------------------------
# 3. REPROJECT, REPAIR, AND DISSOLVE DUPLICATE ZONES
# --------------------------------------------------
print("\nReprojecting geometries...")

zones = zones.to_crs(PROJECTED_CRS)
tracts = tracts.to_crs(PROJECTED_CRS)
roads = roads.to_crs(PROJECTED_CRS)
infrastructure = infrastructure.to_crs(PROJECTED_CRS)

zones = zones[["locationid", "zone", "borough", "geometry"]].copy()
zones["geometry"] = zones.geometry.make_valid()
tracts["geometry"] = tracts.geometry.make_valid()

# FIX: some LocationIDs (e.g. 56, 103) are split over several polygons.
# Merge them so each LocationID is exactly one row. Without this, every
# .reindex()/.map() on locationid fails and areas/counts get double counted.
n_before = len(zones)
zones = zones.dissolve(by="locationid", aggfunc="first").reset_index()
print(f"Zone polygons: {n_before} -> unique LocationIDs: {len(zones)}")

zones["zone_area_km2"] = zones.geometry.area / 1_000_000

tracts = tracts[
    tracts.geometry.notna()
    & ~tracts.geometry.is_empty
    & tracts.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()


# --------------------------------------------------
# 4. CENSUS TRACTS -> TAXI ZONES  (areal interpolation)
# --------------------------------------------------
print("\nMapping Census tracts to taxi zones...")

# The overlay finds every region where a tract and a zone overlap.
intersections = gpd.overlay(
    zones[["locationid", "geometry"]],
    tracts[["GEOID"] + census_columns + ["geometry"]],
    how="intersection",
    keep_geom_type=True,
)
intersections["intersection_area"] = intersections.geometry.area
intersections = intersections[intersections["intersection_area"] > 0].copy()

# FIX: divide by the tract's area INSIDE ZONES, not its full polygon area.
# TIGER tracts include water (rivers, harbour) that no taxi zone covers, so
# the old fraction summed to < 1 and silently lost population.
# Now each tract's fractions sum to 1 and its counts are fully conserved.
tract_total_in_zones = intersections.groupby("GEOID")["intersection_area"].transform("sum")
intersections["tract_fraction"] = intersections["intersection_area"] / tract_total_in_zones

allocation_columns = [
    "B01003_001E", "B17001_001E", "B17001_002E",
    "B08201_001E", "B08201_002E", "B03002_001E", "B03002_003E",
]

for col in allocation_columns:
    intersections[f"{col}_allocated"] = intersections[col] * intersections["tract_fraction"]

socio = (
    intersections.groupby("locationid")[[f"{c}_allocated" for c in allocation_columns]]
    .sum(min_count=1)
    .reset_index()
)

# Sanity check: total population should be (almost) conserved.
pop_in = tracts["B01003_001E"].sum()
pop_out = socio["B01003_001E_allocated"].sum()
print(f"Population check: tracts = {pop_in:,.0f}, allocated to zones = {pop_out:,.0f} "
      f"({pop_out / pop_in:.2%})")


def safe_ratio(num, den):
    return num / den.where(den > 0)


socio["estimated_population"] = socio["B01003_001E_allocated"]
socio["poverty_rate"] = safe_ratio(socio["B17001_002E_allocated"], socio["B17001_001E_allocated"])
socio["zero_vehicle_rate"] = safe_ratio(socio["B08201_002E_allocated"], socio["B08201_001E_allocated"])
socio["minority_share"] = 1 - safe_ratio(socio["B03002_003E_allocated"], socio["B03002_001E_allocated"])

# Income proxy: mean of tract median incomes weighted by allocated HOUSEHOLDS
# (better than area weights: a park has area but no households).
# NOTE: this is a weighted mean of medians, not a true median.
inc = intersections.dropna(subset=["B19013_001E"]).copy()
inc["w"] = inc["B08201_001E_allocated"].fillna(0)
inc["wx"] = inc["B19013_001E"] * inc["w"]
income = inc.groupby("locationid").agg(wx=("wx", "sum"), w=("w", "sum")).reset_index()
income["median_income_proxy"] = income["wx"] / income["w"].where(income["w"] > 0)

socio = socio.merge(income[["locationid", "median_income_proxy"]], on="locationid", how="left")

intersections[["locationid", "GEOID", "intersection_area", "tract_fraction"]].to_csv(
    OUTPUT_CROSSWALK, index=False
)
print("Census mapping completed.")


# --------------------------------------------------
# 5. OSM ROADS -> TAXI ZONES
# --------------------------------------------------
print("\nMapping OSM roads to taxi zones...")

roads = roads[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

# FIX: vectorised clipping. The old row-by-row .apply() ran one geometry
# operation per road (very slow on ~140k segments). overlay does it in bulk
# and cuts roads that cross a zone border into one piece per zone.
clipped = gpd.overlay(
    roads[["geometry"]],
    zones[["locationid", "geometry"]],
    how="intersection",
    keep_geom_type=True,
)
clipped["clipped_length_m"] = clipped.geometry.length

road_stats = (
    clipped.groupby("locationid")["clipped_length_m"].sum()
    .rename("total_road_length_m").reset_index()
)
# FIX: merge instead of .reindex(), which crashed on duplicate IDs.
road_stats = road_stats.merge(zones[["locationid", "zone_area_km2"]], on="locationid", how="left")
road_stats["road_density_km_per_km2"] = (road_stats["total_road_length_m"] / 1000) / road_stats["zone_area_km2"]
road_stats = road_stats.drop(columns="zone_area_km2")
print("Road mapping completed.")


# --------------------------------------------------
# 6. OSM INFRASTRUCTURE -> TAXI ZONES
# --------------------------------------------------
print("\nMapping OSM infrastructure to taxi zones...")

infrastructure = infrastructure[infrastructure.geometry.notna() & ~infrastructure.geometry.is_empty].copy()
infrastructure["geometry"] = infrastructure.geometry.representative_point()

for col in ("amenity", "railway", "public_transport"):
    if col not in infrastructure.columns:
        infrastructure[col] = np.nan

infra_matches = gpd.sjoin(
    infrastructure, zones[["locationid", "geometry"]], how="inner", predicate="within"
)

infra_stats = (
    infra_matches.groupby("locationid")
    .agg(
        total_osm_features=("geometry", "size"),
        hospital_count=("amenity", lambda x: (x == "hospital").sum()),
        school_count=("amenity", lambda x: (x == "school").sum()),
        restaurant_count=("amenity", lambda x: (x == "restaurant").sum()),
        pharmacy_count=("amenity", lambda x: (x == "pharmacy").sum()),
        fuel_station_count=("amenity", lambda x: (x == "fuel").sum()),
        railway_feature_count=("railway", lambda x: x.notna().sum()),
        public_transport_feature_count=("public_transport", lambda x: x.notna().sum()),
    )
    .reset_index()
)
print("Infrastructure mapping completed.")


# --------------------------------------------------
# 7. COMBINE ALL FEATURES
# --------------------------------------------------
print("\nCombining all features...")

final = zones.drop(columns="geometry").copy()
final = final.merge(socio, on="locationid", how="left")
final = final.merge(road_stats, on="locationid", how="left")
final = final.merge(infra_stats, on="locationid", how="left")

osm_columns = [
    "total_road_length_m", "road_density_km_per_km2", "total_osm_features",
    "hospital_count", "school_count", "restaurant_count", "pharmacy_count",
    "fuel_station_count", "railway_feature_count", "public_transport_feature_count",
]
final[osm_columns] = final[osm_columns].fillna(0)

assert final["locationid"].is_unique, "Duplicate LocationIDs in final table"
final.to_csv(OUTPUT_FINAL, index=False)

print("\n======================================")
print("SPATIAL FEATURE ENGINEERING COMPLETED")
print("======================================")
print("Taxi zones in final dataset:", len(final))
print("Final columns:", len(final.columns))
print("Output:", OUTPUT_FINAL)
print("\nSample output:")
print(final.head())
print("\nMissing values:")
print(final.isnull().sum())


Loading datasets...
Taxi zone polygons: 263
Census tracts: 2327
Census records: 2327
OSM roads: 139222
OSM infrastructure: 35658

Preparing Census data...
Tracts with a census match: 2327 / 2327

Reprojecting geometries...
Zone polygons: 263 -> unique LocationIDs: 260

Mapping Census tracts to taxi zones...
Population check: tracts = 8,483,844, allocated to zones = 8,483,844 (100.00%)
Census mapping completed.

Mapping OSM roads to taxi zones...
Road mapping completed.

Mapping OSM infrastructure to taxi zones...
Infrastructure mapping completed.

Combining all features...

SPATIAL FEATURE ENGINEERING COMPLETED
Taxi zones in final dataset: 260
Final columns: 26
Output: data/processed/taxi_zone_features.csv

Sample output:
  locationid              zone    borough  zone_area_km2  \
0          1    Newark Airport        EWR       7.338068   
1         10      Baisley Park     Queens       4.089383   
2        100  Garment District  Manhattan       0.351170   
3        101         Glen O

In [5]:

import os
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# --------------------------------------------------
# 1. PATHS
# --------------------------------------------------

RAW = Path("data/raw")
PROCESSED = Path("data/processed/taxi")

os.makedirs(PROCESSED, exist_ok=True)

# Find the downloaded January 2024 Yellow Taxi file
matches = list(RAW.rglob("yellow_tripdata_2024-01.parquet"))

if not matches:
    raise FileNotFoundError(
        "Could not find yellow_tripdata_2024-01.parquet "
        "inside data/raw"
    )

TAXI_FILE = str(matches[0])

print("Taxi file:", TAXI_FILE)


# --------------------------------------------------
# 2. START SPARK
# --------------------------------------------------

spark = (
    SparkSession.builder
    .appName("NYC_Taxi_Trip_Aggregation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("\nSpark started successfully.")


# --------------------------------------------------
# 3. READ TAXI DATA
# --------------------------------------------------

print("\nReading taxi data...")

trips = spark.read.parquet(TAXI_FILE)

print("Total records:", trips.count())

print("\nAvailable columns:")
trips.printSchema()


# --------------------------------------------------
# 4. SELECT AND CLEAN DATA
# --------------------------------------------------

print("\nCleaning taxi records...")

trips = trips.select(
    "PULocationID",
    "DOLocationID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount"
)

# Convert relevant fields to appropriate types
trips = (
    trips
    .withColumn("PULocationID", F.col("PULocationID").cast("int"))
    .withColumn("DOLocationID", F.col("DOLocationID").cast("int"))
    .withColumn("trip_distance", F.col("trip_distance").cast("double"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn(
        "tpep_pickup_datetime",
        F.to_timestamp("tpep_pickup_datetime")
    )
    .withColumn(
        "tpep_dropoff_datetime",
        F.to_timestamp("tpep_dropoff_datetime")
    )
)

# Remove records with invalid essential fields
trips = trips.filter(
    F.col("PULocationID").isNotNull()
    & F.col("DOLocationID").isNotNull()
    & F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
    & (F.col("trip_distance") > 0)
    & (F.col("fare_amount") >= 0)
    & (F.col("total_amount") >= 0)
)

print("Valid records:", trips.count())


# --------------------------------------------------
# 5. PICKUP-ZONE AGGREGATION
# --------------------------------------------------

print("\nAggregating pickup activity...")

pickup_stats = (
    trips
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("pickup_trips"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("total_amount").alias("avg_total_amount")
    )
    .withColumnRenamed("PULocationID", "locationid")
)

pickup_stats.write.mode("overwrite").parquet(
    str(PROCESSED / "pickup_stats")
)

print("Pickup aggregation completed.")


# --------------------------------------------------
# 6. DROPOFF-ZONE AGGREGATION
# --------------------------------------------------

print("\nAggregating drop-off activity...")

dropoff_stats = (
    trips
    .groupBy("DOLocationID")
    .agg(
        F.count("*").alias("dropoff_trips")
    )
    .withColumnRenamed("DOLocationID", "locationid")
)

dropoff_stats.write.mode("overwrite").parquet(
    str(PROCESSED / "dropoff_stats")
)

print("Drop-off aggregation completed.")


# --------------------------------------------------
# 7. HOURLY PICKUP AGGREGATION
# --------------------------------------------------

print("\nAggregating hourly pickup activity...")

hourly_stats = (
    trips
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime")
    )
    .groupBy("PULocationID", "pickup_hour")
    .agg(
        F.count("*").alias("pickup_trips")
    )
    .withColumnRenamed("PULocationID", "locationid")
)

hourly_stats.write.mode("overwrite").parquet(
    str(PROCESSED / "hourly_pickup_stats")
)

print("Hourly aggregation completed.")


# --------------------------------------------------
# 8. COMBINE PICKUP AND DROPOFF STATISTICS
# --------------------------------------------------

print("\nCombining pickup and drop-off statistics...")

zone_stats = (
    pickup_stats
    .join(dropoff_stats, on="locationid", how="full")
    .fillna(0, subset=["pickup_trips", "dropoff_trips"])
)

zone_stats.write.mode("overwrite").parquet(
    str(PROCESSED / "taxi_zone_trip_stats")
)


# --------------------------------------------------
# 9. DISPLAY RESULTS
# --------------------------------------------------

print("\n======================================")
print("TAXI TRIP AGGREGATION COMPLETED")
print("======================================")

print("\nPickup statistics:")
pickup_stats.orderBy(
    F.desc("pickup_trips")
).show(10)

print("\nDrop-off statistics:")
dropoff_stats.orderBy(
    F.desc("dropoff_trips")
).show(10)

print("\nHourly pickup statistics:")
hourly_stats.orderBy(
    "locationid", "pickup_hour"
).show(10)

print("\nOutput directories:")
print(PROCESSED / "pickup_stats")
print(PROCESSED / "dropoff_stats")
print(PROCESSED / "hourly_pickup_stats")
print(PROCESSED / "taxi_zone_trip_stats")

spark.stop()

print("\nSpark stopped.")


Taxi file: data\raw\tlc\yellow_tripdata_2024-01.parquet

Spark started successfully.

Reading taxi data...
Total records: 2964624

Available columns:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: d

In [7]:

import os
import pandas as pd
from pyspark.sql import SparkSession


# --------------------------------------------------
# 1. PATHS
# --------------------------------------------------

SPATIAL_FILE = "data/processed/taxi_zone_features.csv"

TAXI_STATS = "data/processed/taxi/taxi_zone_trip_stats"

OUTPUT_DIR = "data/processed"
OUTPUT_FILE = f"{OUTPUT_DIR}/final_taxi_zone_dataset.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --------------------------------------------------
# 2. START SPARK
# --------------------------------------------------

spark = (
    SparkSession.builder
    .appName("NYC_Taxi_Feature_Integration")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("\nSpark started successfully.")


# --------------------------------------------------
# 3. LOAD SPATIAL FEATURES
# --------------------------------------------------

print("\nLoading spatial features...")

spatial = pd.read_csv(SPATIAL_FILE)

# Ensure consistent join-key types
spatial["locationid"] = pd.to_numeric(
    spatial["locationid"], errors="coerce"
).astype("Int64")

print("Spatial records:", len(spatial))


# --------------------------------------------------
# 4. LOAD TAXI AGGREGATIONS
# --------------------------------------------------

print("\nLoading taxi statistics...")

taxi = spark.read.parquet(TAXI_STATS)

taxi.printSchema()

taxi_pd = taxi.toPandas()
# Identify taxi LocationIDs without spatial boundaries
spatial_ids = set(spatial["locationid"].dropna())
taxi_ids = set(taxi_pd["locationid"].dropna())

unmatched_ids = taxi_ids - spatial_ids

print("\nUnmatched taxi LocationIDs:")
print(sorted(unmatched_ids))

print("Number of unmatched IDs:", len(unmatched_ids))

print("\nTaxi statistics for unmatched IDs:")

print(
    taxi_pd[
        taxi_pd["locationid"].isin(unmatched_ids)
    ].to_string(index=False)
)

taxi_pd["locationid"] = pd.to_numeric(
    taxi_pd["locationid"], errors="coerce"
).astype("Int64")

print("Taxi aggregation records:", len(taxi_pd))


# --------------------------------------------------
# 5. MERGE DATASETS
# --------------------------------------------------

print("\nIntegrating spatial and taxi features...")

final = spatial.merge(
    taxi_pd,
    on="locationid",
    how="left",
    validate="one_to_one"
)

# Zones with no recorded trips get zero trip counts.
# Missing average fare/distance remain missing.
count_columns = ["pickup_trips", "dropoff_trips"]

for col in count_columns:
    final[col] = final[col].fillna(0).astype("int64")


# --------------------------------------------------
# 6. CALCULATE NORMALIZED METRICS
# --------------------------------------------------

print("\nCalculating normalized activity metrics...")

final["pickups_per_1000_residents"] = (
    final["pickup_trips"]
    / final["estimated_population"].where(
        final["estimated_population"] > 0
    )
    * 1000
)

final["dropoffs_per_1000_residents"] = (
    final["dropoff_trips"]
    / final["estimated_population"].where(
        final["estimated_population"] > 0
    )
    * 1000
)


# --------------------------------------------------
# 7. VALIDATE FINAL DATASET
# --------------------------------------------------

print("\nValidating final dataset...")

assert final["locationid"].is_unique, (
    "Duplicate taxi-zone IDs found!"
)

print("Unique taxi zones:", final["locationid"].nunique())
print("Total columns:", len(final.columns))

print("\nPickup trip total:", final["pickup_trips"].sum())
print("Drop-off trip total:", final["dropoff_trips"].sum())

print("\nMissing values:")
print(final.isnull().sum())


# --------------------------------------------------
# 8. SAVE FINAL DATASET
# ----------------------
# ----------------------------

final.to_csv(OUTPUT_FILE, index=False)

print("\n======================================")
print("DATA INTEGRATION COMPLETED")
print("======================================")

print("Output:", OUTPUT_FILE)

print("\nSample final dataset:")
print(final.head(10).to_string(index=False))

spark.stop()

print("\nSpark stopped.")



Spark started successfully.

Loading spatial features...
Spatial records: 260

Loading taxi statistics...
root
 |-- locationid: integer (nullable = true)
 |-- pickup_trips: long (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_total_amount: double (nullable = true)
 |-- dropoff_trips: long (nullable = true)


Unmatched taxi LocationIDs:
[57, 105, 264, 265]
Number of unmatched IDs: 4

Taxi statistics for unmatched IDs:
 locationid  pickup_trips  avg_trip_distance  avg_fare  avg_total_amount  dropoff_trips
         57             7           4.468571 21.964286         24.632857             46
        105             1           1.410000 11.400000         18.480000              1
        264          9641           3.307271 18.981599         27.347827          14168
        265           705           9.458170 71.051702         81.682468          10803
Taxi aggregation records: 262

Integrating spatial and taxi features..

In [10]:
import geopandas as gpd

zones = gpd.read_file("data/raw/boundaries/taxi_zones.geojson")

present_ids = set(zones["locationid"].astype(int))
expected_ids = set(range(1, 264))

missing_ids = sorted(expected_ids - present_ids)

print("Boundary records:", len(zones))
print("Unique LocationIDs:", len(present_ids))
print("Missing LocationIDs:", missing_ids)


Boundary records: 263
Unique LocationIDs: 260
Missing LocationIDs: [57, 104, 105]


In [14]:
"""
Exploratory Data Analysis for the NYC Yellow Taxi Fairness Mini-Project.

Run from the project root:
    python analyze_fairness.py

Input:
    data/processed/final_taxi_zone_dataset.csv

Outputs:
    reports/eda/summary.txt
    reports/eda/missing_values.csv
    reports/eda/numeric_description.csv
    reports/eda/top_pickup_zones.csv
    reports/eda/lowest_pickup_zones.csv
    reports/eda/correlation_matrix.csv
    reports/eda/*.png
"""

from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "final_taxi_zone_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    print(f"ERROR: Could not find input file: {INPUT_FILE}")
    print("Run this script from your project folder and check the input path.")
    sys.exit(1)

df = pd.read_csv(INPUT_FILE)

print("=" * 65)
print("NYC YELLOW TAXI DATASET - EXPLORATORY DATA ANALYSIS")
print("=" * 65)
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")


def find_column(candidates):
    """Find a column using case-insensitive candidate names."""
    lookup = {str(col).lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


zone_col = find_column(["zone", "zone_name"])
borough_col = find_column(["borough"])
location_col = find_column(["locationid"])
pickup_col = find_column(["pickup_trips", "pickups"])
dropoff_col = find_column(["dropoff_trips", "dropoffs"])
population_col = find_column([
    "estimated_population",
    "population",
    "total_population",
    "allocated_population"
])
income_col = find_column([
    "income_proxy", "median_income_proxy", "median_household_income",
    "income", "weighted_median_income"
])
poverty_col = find_column(["poverty_rate"])
minority_col = find_column(["minority_share"])
zero_vehicle_col = find_column(["zero_vehicle_rate"])
road_density_col = find_column([
    "road_density_km_per_km2",
    "road_density"
])

if pickup_col is None or dropoff_col is None:
    print("ERROR: Could not find pickup_trips and/or dropoff_trips.")
    print("Available columns:", df.columns.tolist())
    sys.exit(1)

print("\nDetected columns:")
for label, col in [
    ("Location ID", location_col), ("Zone", zone_col), ("Borough", borough_col),
    ("Pickup trips", pickup_col), ("Drop-off trips", dropoff_col),
    ("Population", population_col), ("Income proxy", income_col),
    ("Poverty rate", poverty_col), ("Minority share", minority_col),
    ("Zero-vehicle rate", zero_vehicle_col), ("Road density", road_density_col),
]:
    print(f"  {label:20s}: {col if col is not None else 'Not found'}")

# Data quality
missing = df.isna().sum().sort_values(ascending=False)
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (df.isna().mean() * 100).round(2),
})
missing_report.to_csv(OUTPUT_DIR / "missing_values.csv")

numeric_df = df.select_dtypes(include="number")
numeric_df.describe().T.to_csv(OUTPUT_DIR / "numeric_description.csv")

duplicate_count = int(df[location_col].duplicated().sum()) if location_col else 0

# Zone rankings
ranking_cols = [
    col for col in [location_col, zone_col, borough_col, pickup_col, dropoff_col]
    if col is not None
]
top_pickups = df.sort_values(pickup_col, ascending=False)[ranking_cols].head(10)
lowest_pickups = df.sort_values(pickup_col, ascending=True)[ranking_cols].head(10)
top_pickups.to_csv(OUTPUT_DIR / "top_pickup_zones.csv", index=False)
lowest_pickups.to_csv(OUTPUT_DIR / "lowest_pickup_zones.csv", index=False)

# Correlations between available numeric variables
analysis_cols = [
    col for col in [
        pickup_col, dropoff_col, population_col, income_col, poverty_col,
        minority_col, zero_vehicle_col, road_density_col
    ]
    if col is not None and col in numeric_df.columns
]
correlation = df[analysis_cols].corr(numeric_only=True)
correlation.to_csv(OUTPUT_DIR / "correlation_matrix.csv")

# Text report
summary = [
    "NYC YELLOW TAXI - EXPLORATORY DATA ANALYSIS",
    "=" * 55,
    f"Rows: {len(df)}",
    f"Columns: {len(df.columns)}",
    f"Duplicate LocationID rows: {duplicate_count}",
    "",
    "PICKUP TRIPS", str(df[pickup_col].describe().round(2)),
    "",
    "DROP-OFF TRIPS", str(df[dropoff_col].describe().round(2)),
    "",
    "MISSING VALUES", missing_report.to_string(),
    "",
    "TOP 10 PICKUP ZONES", top_pickups.to_string(index=False),
    "",
    "10 LOWEST PICKUP ZONES", lowest_pickups.to_string(index=False),
    "",
    "CORRELATION MATRIX", correlation.round(3).to_string(),
    "",
    "INTERPRETATION NOTE:",
    "Pickup/drop-off counts describe observed taxi trip activity. They do not",
    "directly measure taxi availability, passenger waiting time, rejected",
    "requests, or the probability that a resident can obtain a taxi.",
    "Rates per 1,000 residents should be interpreted cautiously in airports,",
    "commercial districts, tourist areas, and zones with very small populations.",
    "Missing socioeconomic values are not treated as zero.",
]
(OUTPUT_DIR / "summary.txt").write_text("\n".join(summary), encoding="utf-8")


def save_bar_chart(data, category_col, value_col, title, filename):
    plot_data = data.copy()
    if category_col:
        plot_data["_label"] = plot_data[category_col].fillna("Unknown").astype(str)
    elif location_col:
        plot_data["_label"] = plot_data[location_col].astype(str)
    else:
        plot_data["_label"] = plot_data.index.astype(str)

    plot_data = plot_data.sort_values(value_col, ascending=True)
    plt.figure(figsize=(11, 7))
    plt.barh(plot_data["_label"], plot_data[value_col])
    plt.xlabel(value_col.replace("_", " ").title())
    plt.ylabel("Taxi zone")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.close()


label_col = zone_col if zone_col is not None else location_col
save_bar_chart(
    top_pickups, label_col, pickup_col,
    "10 NYC Zones with the Most Yellow Taxi Pickups", "top_10_pickup_zones.png"
)
save_bar_chart(
    lowest_pickups, label_col, pickup_col,
    "10 NYC Zones with the Fewest Yellow Taxi Pickups", "lowest_10_pickup_zones.png"
)

plt.figure(figsize=(10, 6))
plt.hist(df[pickup_col].dropna(), bins=30)
plt.xlabel("Monthly pickup trips")
plt.ylabel("Number of taxi zones")
plt.title("Distribution of Yellow Taxi Pickups Across Zones")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pickup_distribution.png", dpi=200)
plt.close()

if correlation.shape[1] >= 2:
    size = max(8, len(correlation.columns) * 1.1)
    plt.figure(figsize=(size, size))
    image = plt.imshow(correlation, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(image, label="Pearson correlation")
    plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha="right")
    plt.yticks(range(len(correlation.columns)), correlation.columns)
    plt.title("Correlation Between Taxi Activity and Zone Characteristics")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=200)
    plt.close()

for x_col, title, filename in [
    (income_col, "Income Proxy vs Yellow Taxi Pickups", "income_vs_pickups.png"),
    (poverty_col, "Poverty Rate vs Yellow Taxi Pickups", "poverty_vs_pickups.png"),
    (minority_col, "Minority Share vs Yellow Taxi Pickups", "minority_vs_pickups.png"),
    (zero_vehicle_col, "Zero-Vehicle Rate vs Yellow Taxi Pickups", "zero_vehicle_vs_pickups.png"),
]:
    if x_col is not None:
        plot_data = df[[x_col, pickup_col]].dropna()
        if len(plot_data) > 1:
            plt.figure(figsize=(8, 6))
            plt.scatter(plot_data[x_col], plot_data[pickup_col], alpha=0.65)
            plt.xlabel(x_col.replace("_", " ").title())
            plt.ylabel("Monthly pickup trips")
            plt.title(title)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / filename, dpi=200)
            plt.close()

print("\nEDA completed successfully.")
print(f"Reports and charts saved in: {OUTPUT_DIR}")
print("Remember: correlations show associations, not causation.")


NYC YELLOW TAXI DATASET - EXPLORATORY DATA ANALYSIS
Dataset shape: 260 rows x 33 columns

Detected columns:
  Location ID         : locationid
  Zone                : zone
  Borough             : borough
  Pickup trips        : pickup_trips
  Drop-off trips      : dropoff_trips
  Population          : estimated_population
  Income proxy        : median_income_proxy
  Poverty rate        : poverty_rate
  Minority share      : minority_share
  Zero-vehicle rate   : zero_vehicle_rate
  Road density        : road_density_km_per_km2

EDA completed successfully.
Reports and charts saved in: c:\Users\Jayanth\Downloads\5th sem\BD\Mini-project\reports\eda
Remember: correlations show associations, not causation.


In [15]:
display(correlation.round(3))

,pickup_trips,dropoff_trips,estimated_population,median_income_proxy,poverty_rate,minority_share,zero_vehicle_rate,road_density_km_per_km2
pickup_trips,1.000,0.941,-0.202,0.477,-0.211,-0.357,0.340,-0.112
dropoff_trips,0.941,1.000,-0.136,0.526,-0.227,-0.401,0.432,-0.043
estimated_population,-0.202,-0.136,1.000,-0.316,0.204,0.285,0.036,0.349
median_income_proxy,0.477,0.526,-0.316,1.000,-0.704,-0.685,0.080,-0.054
poverty_rate,-0.211,-0.227,0.204,-0.704,1.000,0.583,0.424,0.039
minority_share,-0.357,-0.401,0.285,-0.685,0.583,1.000,0.074,0.237
zero_vehicle_rate,0.340,0.432,0.036,0.080,0.424,0.074,1.000,0.034
road_density_km_per_km2,-0.112,-0.043,0.349,-0.054,0.039,0.237,0.034,1.000


In [19]:
display(
    df[
        [
            "zone",
            "borough",
            "estimated_population",
            "pickup_trips",
            "pickups_per_1000_residents"
        ]
    ].sort_values(
        "pickups_per_1000_residents",
        ascending=False
    ).head(15)
)

,zone,borough,estimated_population,pickup_trips,pickups_per_1000_residents
199,Central Park,Manhattan,16.248398,47335,2.913210e+06
35,JFK Airport,Queens,84.733299,138491,1.634434e+06
2,Garment District,Manhattan,344.350733,46664,1.355130e+05
41,LaGuardia Airport,Queens,741.751976,87712,1.182498e+05
67,Midtown Center,Manhattan,1956.363962,140156,7.164107e+04
68,Midtown East,Manhattan,4040.086417,104354,2.582965e+04
94,Penn Station/Madison Sq West,Manhattan,5629.534069,102160,1.814715e+04
21,Battery Park,Manhattan,56.125761,846,1.507329e+04
144,Times Sq/Theatre District,Manhattan,7541.308466,102968,1.365386e+04
69,Midtown North,Manhattan,7956.532756,83690,1.051840e+04


In [20]:
display(top_pickups)

,locationid,zone,borough,pickup_trips,dropoff_trips
67,161,Midtown Center,Manhattan,140156,109269
151,237,Upper East Side South,Manhattan,140131,127670
35,132,JFK Airport,Queens,138491,27485
150,236,Upper East Side North,Manhattan,133974,139418
68,162,Midtown East,Manhattan,104354,83151
144,230,Times Sq/Theatre District,Manhattan,102968,87648
94,186,Penn Station/Madison Sq West,Manhattan,102160,62227
46,142,Lincoln Square East,Manhattan,101800,87620
41,138,LaGuardia Airport,Queens,87712,30117
153,239,Upper West Side South,Manhattan,86475,87235
